### Multiple Linear Regression 

#### House price prediction

##### Import packages

In [1]:
import pandas as pd 
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

In [2]:
df = pd.read_csv('housing_price_dataset.csv')
df.sample(5)

,SquareFeet,Bedrooms,Bathrooms,Neighborhood,YearBuilt,Price
8134,1507,3,2,Urban,2015,210833.643354
29292,1693,5,1,Rural,1985,174345.879425
40267,2168,5,1,Rural,1979,275536.428576
44256,2907,3,2,Suburb,1980,372638.566251
18303,1775,4,2,Rural,2010,256696.082682


##### Understand the data types and features

In [3]:
df.describe()

,SquareFeet,Bedrooms,Bathrooms,YearBuilt,Price
count,50000.000000,50000.000000,50000.000000,50000.000000,50000.000000
mean,2006.374680,3.498700,1.995420,1985.404420,224827.325151
std,575.513241,1.116326,0.815851,20.719377,76141.842966
min,1000.000000,2.000000,1.000000,1950.000000,-36588.165397
25%,1513.000000,3.000000,1.000000,1967.000000,169955.860225
50%,2007.000000,3.000000,2.000000,1985.000000,225052.141166
75%,2506.000000,4.000000,3.000000,2003.000000,279373.630052
max,2999.000000,5.000000,3.000000,2021.000000,492195.259972


In [4]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 50000 entries, 0 to 49999
Data columns (total 6 columns):
 #   Column        Non-Null Count  Dtype  
---  ------        --------------  -----  
 0   SquareFeet    50000 non-null  int64  
 1   Bedrooms      50000 non-null  int64  
 2   Bathrooms     50000 non-null  int64  
 3   Neighborhood  50000 non-null  object 
 4   YearBuilt     50000 non-null  int64  
 5   Price         50000 non-null  float64
dtypes: float64(1), int64(4), object(1)
memory usage: 2.3+ MB


In [5]:
# is there any missing value?
df.isnull().sum()

SquareFeet      0
Bedrooms        0
Bathrooms       0
Neighborhood    0
YearBuilt       0
Price           0
dtype: int64

In [6]:
# handle categorical columns
df['Neighborhood'].value_counts()

Neighborhood
Suburb    16721
Rural     16676
Urban     16603
Name: count, dtype: int64

#### Extracting the features

In [7]:
# drop the 'year' column 
df.drop(columns=['YearBuilt'],inplace=True)

In [8]:
X = df.drop(columns=['Price'])
y = df['Price']

print(X)
print(y)

       SquareFeet  Bedrooms  Bathrooms Neighborhood
0            2126         4          1        Rural
1            2459         3          2        Rural
2            1860         2          1       Suburb
3            2294         2          1        Urban
4            2130         5          2       Suburb
...           ...       ...        ...          ...
49995        1282         5          3        Rural
49996        2854         2          2       Suburb
49997        2979         5          3       Suburb
49998        2596         5          2        Rural
49999        1572         5          3        Rural

[50000 rows x 4 columns]
0        215355.283618
1        195014.221626
2        306891.012076
3        206786.787153
4        272436.239065
             ...      
49995    100080.865895
49996    374507.656727
49997    384110.555590
49998    380512.685957
49999    221618.583218
Name: Price, Length: 50000, dtype: float64


#### Normalize features

In [9]:
from sklearn.model_selection import train_test_split    

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)


In [10]:
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.compose import ColumnTransformer

numeric_features = ['SquareFeet', 'Bedrooms', 'Bathrooms']
categorical_features = ['Neighborhood']

preprocessor = ColumnTransformer(
    transformers=[
        ('num', StandardScaler(), numeric_features),
        ('cat', OneHotEncoder(), categorical_features)
    ])

X_train = preprocessor.fit_transform(X_train)
X_test = preprocessor.transform(X_test)

#### Model training 

In [11]:
# LinearRegression = Closed-form solution via Normal Equation / SVD, does not scale well for large datasets.It does not use gradient descent

from sklearn.linear_model import SGDRegressor

sgd_model = SGDRegressor(loss="squared_error", max_iter=1000, learning_rate="invscaling", eta0=0.01)
sgd_model.fit(X_train, y_train)

print("Coefficients:", sgd_model.coef_)
print("Intercept:", sgd_model.intercept_)




Coefficients: [57411.21527527  5625.80482429  2361.92184636 54739.97537745
 53781.07798866 56189.2013109 ]
Intercept: [169252.48493544]


- loss="squared_error" → minimizes MSE
- eta0 → initial learning rate
- max_iter → number of iterations
- learning_rate="invscaling"
    - Early iterations: larger steps → faster learning.
    - Later iterations: smaller steps → more stable convergence.
    - Helps avoid overshooting the minimum of the cost function.


In [12]:
X_test

array([[-0.19463463,  1.34247591, -1.22207152,  1.        ,  0.        ,
         0.        ],
       [-1.74706239,  1.34247591,  1.22893434,  0.        ,  1.        ,
         0.        ],
       [ 0.44858852,  0.4456385 ,  1.22893434,  0.        ,  1.        ,
         0.        ],
       ...,
       [ 1.52816034, -0.4511989 ,  0.00343141,  0.        ,  0.        ,
         1.        ],
       [-0.57709164,  1.34247591,  0.00343141,  0.        ,  0.        ,
         1.        ],
       [ 0.38774308, -0.4511989 ,  0.00343141,  1.        ,  0.        ,
         0.        ]], shape=(10000, 6))

In [13]:
y_pred = sgd_model.predict(X_test)

In [14]:
from sklearn.metrics import r2_score, mean_absolute_error, root_mean_squared_error  


rmse = root_mean_squared_error(y_test,y_pred)
mae = mean_absolute_error(y_test, y_pred)
r2 = r2_score(y_test, y_pred)


print(rmse)
print(mae)
print(r2)
print(rmse/mae)


49361.094765779446
39432.384828033275
0.5755161195900256
1.2517907547576896


In [15]:
# is the model underfitting or overfitting?
y_pred_train = sgd_model.predict(X_train)

r2_score_train = r2_score(y_train, y_pred_train)
r2_score_test = r2_score(y_test, y_pred)

print("train:", r2_score_train)
print("test:", r2_score_test)

# train ~ test => good model
# train >> test => overfitting  
# train << test => underfitting

train: 0.568793331581212
test: 0.5755161195900256
